# Engenharia de Features

Técnicas avançadas de transformação e seleção de features.

**Conceitos:**
- Features polinomiais (`PolynomialFeatures`)
- Seleção de features (`SelectKBest`, `RFE`)
- Transformação de distribuições (`PowerTransformer`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import PowerTransformer

In [ ]:
# Criar dados
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.rand(m, 1)

# Tentativa de regressão linear
lin_reg = LinearRegression()
lin_reg.fit(X, y)
X_novo = np.linspace(-3, 3, 100).reshape(100, 1)
y_novo = lin_reg.predict(X_novo)

In [ ]:
# Engenhariade feaures: criar polinomios
poly_feat = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly_feat.fit_transform(X)

print(f'X original: {X[0]}')
print(f'X transformado: {X_poly[0]}')

In [ ]:
# Treinar no dado transformado
lin_reg_poly = LinearRegression()
lin_reg_poly.fit(X_poly, y)

X_novo_poly = poly_feat.fit_transform(X_novo)
y_novo_poly = lin_reg_poly.predict(X_novo_poly)

In [ ]:
# Visualização
plt.scatter(X, y, c='blue', label='Dados reais')
plt.plot(X_novo, y_novo, 'r--', label='Regressão Linear(reta)', linewidth=2)
plt.plot(X_novo, y_novo_poly, 'g-', label='Regressão Polinomial(curva)', linewidth=2)
plt.legend()
plt.show

In [ ]:
# Criar dataset
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=3,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

# Separar os dados
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)
print(f'Formato original: {X_treino.shape}')

In [ ]:
select = SelectKBest(score_func=f_classif, k=5)

X_treino_select = select.fit_transform(X_treino, y_treino)
X_teste_select = select.transform(X_teste)

print(f'Formado após a seleção: {X_treino_select.shape}')

colunas_select = select.get_support()
print(f'Indice das colunas: {[i for i, x in enumerate(colunas_select) if x]}')

In [ ]:
# O RFE precisa de um modelo base para julgar a importância
modelo_base = RandomForestClassifier(n_estimators=100, random_state=42)

# Selecionar as melhores colunas
rfe = RFE(estimator=modelo_base, n_features_to_select=3)
rfe.fit(X_treino, y_treino)

print('Colunas escolhidas pelo RFE:')
# Ranking: 1 significa que foi selecionada
for i, col in enumerate(range(X_treino.shape[1])):
  print(f'Coluna {col}: Rank: {rfe.ranking_[i]} {'(Selecionada)' if rfe.support_[i] else ""}')

In [ ]:
# Gerar dados log-normais (tortos)
X_torto = np.random.lognormal(mean=0, sigma=1, size=(1000, 1))

# Transformar para ficar parecido com uma Gaussiana
pt = PowerTransformer(method='yeo-johnson')
X_normal = pt.fit_transform(X_torto)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(X_torto, bins=30)
ax[0].set_title('Antes (torto/skewed)')
ax[1].hist(X_normal, bins=30)
ax[1].set_title('Depois (Yeo-Johnson)')
plt.show()